# Lab 2 - MMALS geometric path memory

This notebook translates the geometry into a controlled continual-learning abstraction. A batch of latent features is summarized by a covariance operator

$$\rho = \frac{\Sigma + \epsilon I}{\operatorname{Tr}(\Sigma + \epsilon I)}.$$

This is a **density-like engineering representation**, not a claim that the neural network is a quantum system.

In [ ]:
from pathlib import Path
import sys
repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

import numpy as np
import matplotlib.pyplot as plt

from mmals_path_memory.geometry import normalize_psd, trace_distance, von_neumann_entropy
from mmals_path_memory.mmals import covariance_density, diagnose_transition

## 1. Generate two latent regimes

In [ ]:
rng = np.random.default_rng(42)
base = rng.multivariate_normal([0, 0], [[1.0, 0.45], [0.45, 0.50]], size=800)
rot = np.array([[np.cos(0.65), -np.sin(0.65)], [np.sin(0.65), np.cos(0.65)]])
reoriented = base @ rot.T
structural = reoriented @ np.array([[1.25, 0.30], [0.0, 0.55]]).T

rho_base = covariance_density(base)
rho_reoriented = covariance_density(reoriented)
rho_structural = covariance_density(structural)

print("Base spectrum:", np.linalg.eigvalsh(rho_base))
print("Reoriented spectrum:", np.linalg.eigvalsh(rho_reoriented))
print("Structural spectrum:", np.linalg.eigvalsh(rho_structural))

## 2. Diagnose the transitions

In [ ]:
for name, target in [("reorientation", rho_reoriented), ("structural adaptation", rho_structural)]:
    result = diagnose_transition(rho_base, target)
    print()
    print(name)
    print(result)


## 3. Visualize feature clouds

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(base[::8,0], base[::8,1], alpha=0.5, label="base")
plt.scatter(reoriented[::8,0], reoriented[::8,1], alpha=0.5, label="reoriented")
plt.scatter(structural[::8,0], structural[::8,1], alpha=0.5, label="structural")
plt.axis("equal")
plt.grid(True)
plt.legend()
plt.title("Latent regimes and transformations")
plt.show()

## 4. Curriculum-order experiment

Apply the same two linear adaptations in different orders. The endpoint difference is an order-sensitive proxy for interference.

In [ ]:
A = np.array([[np.cos(0.55), -np.sin(0.55)], [np.sin(0.55), np.cos(0.55)]])
B = np.array([[1.18, 0.28], [0.0, 0.68]])

ab = base @ A.T @ B.T
ba = base @ B.T @ A.T
rho_ab = covariance_density(ab)
rho_ba = covariance_density(ba)

print("Order gap, trace distance:", trace_distance(rho_ab, rho_ba))
print("Entropy A -> B:", von_neumann_entropy(rho_ab))
print("Entropy B -> A:", von_neumann_entropy(rho_ba))
print("Map commutator norm:", np.linalg.norm(A @ B - B @ A))

## 5. Candidate MMALS control rule

A research controller may combine routing energy, uncertainty, replay risk, and transverse geometry:

$$R_t = w_E E_t + w_U U_t + w_T\frac{\|\Delta\rho_t^\perp\|}{\|\Delta\rho_t\|+\epsilon} + w_O O_t.$$

Possible decisions:

- low transverse ratio: adapt current host;
- high transverse ratio and low similarity: create/split host;
- high transverse ratio and known prototype match: reroute or merge;
- high order gap: protect replay and evaluate curriculum robustness.

In [ ]:
result = diagnose_transition(rho_base, rho_structural, transverse_threshold=0.35)
router_energy = 0.22
uncertainty = 0.31
order_gap = trace_distance(rho_ab, rho_ba)
score = 0.25*router_energy + 0.20*uncertainty + 0.40*result.transverse_ratio + 0.15*order_gap
print("Illustrative control score:", score)
print("Recommendation:", result.recommendation)

## Validation protocol

Run the order-permutation matrix on RotatedMNIST, PermutedMNIST, SplitCIFAR-10/100, and CORe50. Compare predictive value for future forgetting against MMD, Fisher-Rao/Bures distance, router energy, and representation-similarity baselines. Use ablations with permuted clusters and oracle regimes.